In [ ]:
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib as mpl
import matplotlib.patheffects as mpe
import matplotlib.pyplot as plt
import numpy as np
import optuna
from pathlib import Path
import pandas as pd
import shapely
import os
import rasterio as rio
from glob import glob
from scipy.stats import mannwhitneyu
import seaborn as sns
import torch
from torch_geometric.utils import k_hop_subgraph
from torch_sparse import SparseTensor
from tqdm import tqdm
import xarray as xr

from project.utils.data import enrich, load_hillshade, sel_subset
from project.utils.metrics import compute_metrics
from project.utils.plotting import ProjectCustomExtent, despine, cbar_halo

plt.rcParams["figure.dpi"] = 150

# Switzerland LV95 with extended limits to cover 'Hydrological Switzerland'.
lv95 = ProjectCustomExtent(2056, [2474273, 2842000, 1058000, 1325000])

In [ ]:
# Load predictions.
ds = xr.open_zarr('/net/argon/landclim/kraftb/machflow_routing/data/results/predictions.zarr').sel(
    time=slice('1993', None), model='routed_random')

# Load HRU polygons.
hrus = gpd.read_file(
    '/net/argon/landclim/kraftb/machflow_routing/data/processed/hrus/hrus.gpkg', use_arrow=True).set_index('EZGNR')

# rivers = gpd.read_file('/net/argon/landclim/kraftb/machflow_routing/data/raw/EZG_Gewaesser.gpkg', layer='VorfluterAbschnitt', use_arrow=True).set_index('EZGNR')

rivers = gpd.read_file('/net/argon/landclim/kraftb/machflow_routing/data/raw/rivernet', use_arrow=True)

rivers_subset = rivers.loc[rivers.CUM_LEN > 20000]

In [ ]:
hrus_with_obs = hrus.loc[ds.hru.where(ds.has_obs.load(), drop=True).values]

In [ ]:
bg = load_hillshade(
    hillshade_dir='/net/argon/landclim/kraftb/machflow_routing/data/processed/',
    hrus=hrus,
    ds=ds,
    use_terrain=True,
    target_crs='EPSG:2056',
    lake_rgb=(86, 150, 245),
    force_recompute=False)

bgw = load_hillshade(
    hillshade_dir='/net/argon/landclim/kraftb/machflow_routing/data/processed/',
    hrus=hrus,
    ds=ds,
    use_terrain=False,
    target_crs='EPSG:2056',
    lake_rgb=(86, 150, 245),
    force_recompute=False)

bounds = bg.rio.bounds()
hydro_ch = hrus.unary_union
hydro_ch = gpd.GeoDataFrame(
    geometry=[hydro_ch],
    crs=hrus.crs        # important: keep same CRS
)
world_box = gpd.GeoDataFrame(
    geometry=[shapely.geometry.box(*bounds)],
    crs=hrus.crs
)
outside = gpd.overlay(world_box, hydro_ch, how="difference")

## HRU map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={'projection': lv95}, gridspec_kw={'wspace': 0.02})

ax.set_global()
ax.set(zorder=1000)

# bodr = cartopy.feature.NaturalEarthFeature(category='cultural', 
#     name='admin_0_boundary_lines_land', scale='10m', facecolor='none')
# ax.add_feature(bodr, linestyle='--', edgecolor='w', alpha=1, zorder=10, lw=0.7)

bg.plot.imshow(ax=ax, interpolation='bilinear', alpha=0.6, zorder=0)
outside.plot(ax=ax, color='w', alpha=0.4, ec='none', lw=0, zorder=0)

scale_axis = ax.inset_axes([0.85, 0.02, 0.1, 0.04], transform=ax.transAxes)
scale_axis.plot([2390000, 2890000], [1020000, 1020000], color='k', lw=2, transform=scale_axis.transData)
scale_axis.text(0.5, 0.54, '50 km', ha='center', va='bottom', fontsize=8, transform=scale_axis.transAxes)
scale_axis.set_axis_off()

hrus.plot(ec='k', fc='none', lw=0.12, zorder=5, ax=ax)
rivers_subset.plot(
    ec='skyblue', fc='none', linewidth=(np.log(rivers_subset.CUM_LEN + 1) / 12 - 0.5) + 0.,
    alpha=1.0,
    zorder=5,
    ax=ax)
# rivers_subset.plot(
#     ec='royalblue', fc='none', linewidth=(np.log(rivers_subset.CUM_LEN + 1) / 12 - 0.5),
#     alpha=1.0,
#     zorder=5,
#     ax=ax)

ax.set_title('')

axins = ax.inset_axes((-0.045, 0.67, 0.32, 0.32), projection=lv95, zorder=20)
axins.set_extent([2610000, 2650000, 1150000, 1195000], crs=lv95)
axins.spines['geo'].set_linewidth(2.0)
ax.indicate_inset_zoom(axins, edgecolor='k', zorder=100, alpha=1.0)
ax.scatter(hrus_with_obs.geometry.centroid.x, hrus_with_obs.geometry.centroid.y,
           s=7, fc="#ff3d3d", edgecolor='k', lw=0.4, zorder=15)

bg.plot.imshow(ax=axins, interpolation='bilinear', alpha=0.6, zorder=20)
# hrus.plot(ec='k', fc='none', lw=0.4, zorder=25, ax=axins)
# rivers.plot(ec="#e3f1ff", fc='none', lw=0.6, zorder=26, ax=axins, alpha=0.7)


hrus.plot(ec='k', fc='none', lw=0.2, zorder=25, ax=axins)
rivers_subset.plot(
    ec='skyblue', fc='none', linewidth=(np.log(rivers_subset.CUM_LEN + 1) / 12 - 0.5) + 0.5,
    alpha=1.0,
    zorder=25,
    ax=axins)
# rivers_subset.plot(
#     ec='k', fc='none', linewidth=(np.log(rivers_subset.CUM_LEN + 1) / 12 - 0.5),
#     alpha=0.8,
#     zorder=25,
#     ax=axins)

axins.scatter(hrus_with_obs.geometry.centroid.x, hrus_with_obs.geometry.centroid.y,
           s=7, fc="#ff3d3d", edgecolor='k', lw=0.4, zorder=30)
axins.set_title('')

fig.savefig('./figures/hru_map.png', dpi=300, bbox_inches='tight')


## Catchment

In [ ]:
edge_index = torch.tensor(np.asarray(
    [[i, np.argwhere(ds.hru.values == tgt).item()] for i, tgt in enumerate(ds.downstream.values) if tgt != -1]).T)

In [ ]:
def find_upstream(hru_i, visited=None):
    if visited is None:
        visited = set()

    # Don't add or process this node again if it's already been visited
    if hru_i in visited:
        return []

    # Mark current node as visited to avoid infinite reprocessing
    visited.add(hru_i)

    # Step 1: Find all direct upstream nodes
    edge_indices = torch.nonzero(edge_index[1] == hru_i, as_tuple=False).flatten()
    direct_upstream = [edge_index[0, idx].item() for idx in edge_indices]

    ds_upstream = ds.isel(hru=direct_upstream)
    direct_upstream = [direct_upstream[i] for i in np.argsort(ds_upstream.x.values).tolist()[-1::-1]]

    results = []

    # Step 2: Add all direct upstream nodes first
    for node in direct_upstream:
        if node not in visited:
            results.append(node)

    # Step 3: Then recurse into each direct upstream node
    for node in direct_upstream:
        results.extend(find_upstream(node, visited))

    return results

hru = 165275  # 116072
hru_i = np.argwhere(ds.hru.values == hru).item()
subset = [hru_i] + find_upstream(hru_i)
subset.reverse()

In [ ]:
hrus_c = hrus.loc[ds.hru.values[subset]]
# order = hrus_c.bounds.maxx.argsort().values
ds_c = ds.sel(hru=hrus_c.index)
rivers_c = rivers.loc[rivers.index.intersection(hrus_c.index)]
hrus_c = enrich(ds_c.R_pred.isel(time=-1) * 100, hrus_c, 'retention')

for var in ds.data_vars:
    if ds[var].dims == ('hru',):
        hrus_c = enrich(ds_c[var], hrus_c, var)

hrus_c = enrich(ds_c['S_pred_m3'].isel(time=10) + ds_c['q_pred_m3s'].isel(time=10) - ds_c['e_pred_m3s'].isel(time=10), hrus_c, 'S_prime')
hrus_c = enrich(ds_c['S_pred_m3'].isel(time=11), hrus_c, 'S')
hrus_c = enrich(ds_c['Q_pred_m3s'].isel(time=11), hrus_c, 'Q')
hrus_c = enrich(ds_c['e_pred_m3s'].isel(time=11), hrus_c, 'e')
hrus_c = enrich(ds_c['q_pred_m3s'].isel(time=11), hrus_c, 'r')

In [ ]:
edge_index_parts = []

for i, hru in enumerate(ds_c.hru.values):
    if i < len(ds.hru):
        hru_down = ds_c.downstream.isel(hru=i).load().item()
        if hru_down in ds_c.hru:
            edge_index_parts.append([i, np.argwhere(ds_c.hru.values == hru_down).item()])
            
edge_index_c = torch.tensor(edge_index_parts).T

In [ ]:
ms = [
    SparseTensor(
        row=torch.arange(len(subset)),
        col=torch.arange(len(subset)),
        value=torch.zeros(len(subset)),
        sparse_sizes=(len(subset), len(subset))
    )
]

m = SparseTensor(
    row=edge_index_c[1, :],
    col=edge_index_c[0, :],
    value=torch.ones(edge_index_c.shape[1]),
    sparse_sizes=(len(subset), len(subset))
)

for i in range(1, 7):
    m_new = m @ ms[i-1]
    r, c, v = m_new.coo()
    v += 1
    ms.append(SparseTensor(row=r, col=c, value=v, sparse_sizes=(len(subset), len(subset))))
    
rows = []
cols = []
ks = []

for m in ms:
    r, c, v = m.coo()
    rows.append(r)
    cols.append(c)
    ks.append(v)

rows = torch.cat(rows)
cols = torch.cat(cols)
ks = torch.cat(ks).long()

T_sparse = SparseTensor(
    row=rows,
    col=cols,
    value=torch.Tensor(ds_c.T_down.values[cols, ks]),
    sparse_sizes=(len(subset), len(subset)))

D_sparse = SparseTensor(
    row=rows,
    col=cols,
    value=torch.Tensor(ds_c.D_down.values[cols, ks]),
    sparse_sizes=(len(subset), len(subset)))

T = T_sparse.to_dense()
D = D_sparse.to_dense()

# T[T == 0] = np.nan
# D[D == 0] = np.nan


In [ ]:
fig = plt.figure(figsize=(15, 5))

gs = mpl.gridspec.GridSpec(2, 8, left=0.05, right=0.48, wspace=0.05)

# Map
# ------------------------

ax0 = fig.add_subplot(gs[0, :3])

axins = ax0.inset_axes((-0.1, 0.7, 0.3, 0.02))

norm = mpl.colors.Normalize(vmin=0, vmax=60)
cmap = 'RdPu'
im = hrus_c.plot(
    column='retention', ax=ax0, ec='k', lw=0.8, legend=False, cmap=cmap, norm=norm)

mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
cbar = plt.colorbar(
    mappable,
    cax=axins,
    orientation='horizontal',   # or 'vertical'
    extend='max',
    label='Retention (%)'
)
cbar.ax.xaxis.label.set_size(8)
cbar.ax.tick_params(labelsize=8)

rivers_c.plot(ax=ax0, lw=1, color='tab:blue')
lake = hrus_c[ds_c.is_lake.values]
lake_x = lake.geometry.centroid.x.item()
lake_y = lake.geometry.centroid.y.item()

ax0.annotate('Lake', color='k', xy=(lake_x, lake_y), xytext=(lake_x + 1000, lake_y - 1000), size=8,
            arrowprops=dict(arrowstyle="-", color='tab:red',
                            connectionstyle="angle3,angleA=0,angleB=-90"))

x = rivers_c.geometry.bounds.maxx.max()
y = rivers_c.geometry.bounds.maxy.max()
ax0.scatter([x], [y], zorder=10, ec='k', fc='w', s=30, lw=0.8)
ax0.annotate('Gauging station', color='k', xy=(x, y), xytext=(x - 3500, y + 650), size=8,
            arrowprops=dict(arrowstyle="-", color='tab:red',
                            connectionstyle="angle3,angleA=0,angleB=90"))

for x, y, name in zip(
        hrus_c.centroid.x,
        hrus_c.centroid.y,
        [chr(ord('a') + i) for i in range(len(hrus_c.centroid.y))]
        #[i for i in range(len(hrus_c.centroid.y))]
    ):
    ax0.text(x, y, name, zorder=100, color='k', fontsize=8, va='center', ha='center',
            bbox=dict(alpha=0.7, boxstyle='circle,pad=0.1', ec='k', fc='w'))

ax0.set_axis_off()

artists = [
    mpl.lines.Line2D([0], [0], color='tab:blue', lw=2, path_effects=[mpe.withStroke(linewidth=3, foreground='w')]),
    mpl.patches.Patch(facecolor='white', edgecolor='k', lw=1)
]
labels= ['River', 'Hydrological\nresponse\nunit (HRU)']

ax0.legend(artists, labels, loc='upper left', frameon=False, fontsize=8, bbox_to_anchor=(-0.15, 1.1))

# Vectors
# ------------------------

ax1 = fig.add_subplot(gs[0, -3:])

for i, (var, var_nice) in enumerate(zip(
        ['sum', 'S_pred_m3', 'q_pred_m3s', 'e_pred_m3s'],
        ['$S\'_t$', '$S_t$', '$q_t$', '$e_t$'])
):

    if var == 'sum':
        ds_sel = ds_c[['S_pred_m3', 'q_pred_m3s', 'e_pred_m3s']].isel(time=-2).to_array().sum(dim='variable').values.reshape(-1, 1)
    else:
        ds_sel = ds_c[var].isel(time=-2).values.reshape(-1, 1)

    operator = ['=', '+', '-', ''][i]

    xloc = i * 0.27

    axins = ax1.inset_axes((xloc, 0.0, 0.23, 1.0))

    im = axins.imshow(ds_sel, vmin=0, vmax=0.7)
    axins.set_xticks([])
    axins.set_yticks(np.arange(len(hrus_c)))
    axins.set_yticklabels([chr(ord('a') + i) for i in range(len(hrus_c))])

    axins.set_title(var_nice, fontsize=8)
    ax1.text(xloc + 0.2, 0.5, operator, fontsize=12, ha='center', va='center', transform=ax1.transAxes)

    if i == 0:
        axins.annotate(
            'State before transport', color='k', xy=(0, 3), xytext=(-7, 0), size=8,
            horizontalalignment='center', arrowprops=dict(arrowstyle="-", color='tab:red',
            connectionstyle="angle3,angleA=-10,angleB=-45"))

# fig.colorbar(im, ax=ax1, orientation='horizontal', label=f'{name} (%)', pad=0.01, shrink=0.5)

ax1.set_axis_off()

# Matrices
# ------------------------

s_prime = ds_c[['S_pred_m3', 'q_pred_m3s', 'e_pred_m3s']].isel(time=-2).to_array().sum(dim='variable').values.reshape(-1, 1)
s_new = ds_c['S_pred_m3'].isel(time=-1).values.reshape(-1, 1)
Q_new = ds_c['Q_pred_m3s'].isel(time=-1).values.reshape(-1, 1)

ax2 = fig.add_subplot(gs[1, :])

ax_s_new = ax2.inset_axes((0.0, 0.0, 0.03, 1.0))
ax_s_new.text(0.03, 0.5, '$=$', transform=ax2.transAxes)
ax_t = ax2.inset_axes((0.08, 0.0, 0.28, 1.0))
ax_s_prime_0 = ax2.inset_axes((0.41, 0.0, 0.03, 1.0))

ax_q_new = ax2.inset_axes((0.55, 0.0, 0.03, 1.0))
ax_s_new.text(0.58, 0.5, '$=$', transform=ax2.transAxes)
ax_d = ax2.inset_axes((0.63, 0.0, 0.28, 1.0))
ax_s_prime_1 = ax2.inset_axes((0.96, 0.0, 0.03, 1.0))

# S t+1
# ------------------------

im = ax_s_new.imshow(s_new, vmin=0, vmax=0.7)
ax_s_new.set_xticks([])
ax_s_new.set_yticks(np.arange(len(hrus_c)))
ax_s_new.set_yticklabels([chr(ord('a') + i) for i in range(len(hrus_c))])
ax_s_new.set_title('$\mathbf{S}_{t+1}$', fontsize=8)

# S'
# ------------------------
im = ax_s_prime_0.imshow(s_prime, vmin=0, vmax=0.7)
ax_s_prime_0.set_xticks([])
ax_s_prime_0.set_yticks(np.arange(len(hrus_c)))
ax_s_prime_0.set_yticklabels([chr(ord('a') + i) for i in range(len(hrus_c))])
ax_s_prime_0.set_title('$\mathbf{S}\'_t$', fontsize=8)

im = ax_s_prime_1.imshow(s_prime, vmin=0, vmax=0.4)
ax_s_prime_1.set_xticks([])
ax_s_prime_1.set_yticks(np.arange(len(hrus_c)))
ax_s_prime_1.set_yticklabels([chr(ord('a') + i) for i in range(len(hrus_c))])
ax_s_prime_1.set_title('$\mathbf{S}\'_t$', fontsize=8)

# Q t+1
# ------------------------
im = ax_q_new.imshow(Q_new, vmin=0, vmax=0.7)
ax_q_new.set_xticks([])
ax_q_new.set_yticks(np.arange(len(hrus_c)))
ax_q_new.set_yticklabels([chr(ord('a') + i) for i in range(len(hrus_c))])
ax_q_new.set_title('$\mathbf{Q}_{t+1}$', fontsize=8)


for i, (ax, data, name, math, cmap) in enumerate(
        zip([ax_t, ax_d], [T, D], ['Transport', 'Discharge'], ['$\mathbf{T}_t$', '$\mathbf{D}_t$'], ['RdPu', 'GnBu'])
    ):
    # Compute proper imshow extent
    nrows, ncols = data.shape
    extent = [-0.5, ncols - 0.5, -0.5, nrows - 0.5]
    bgr = mpl.patches.Rectangle(
        (extent[0], extent[2]),  # bottom-left corner
        ncols,                    # width
        nrows,                    # height
        hatch='///////',
        fill=False,
        edgecolor='0.8',
        linewidth=0.1,
        zorder=-1
    )
    ax.add_patch(bgr)

    im = ax.imshow(data * 100, interpolation='none', cmap=cmap, vmin=0)
    # fig.colorbar(im, ax=ax, orientation='horizontal', label=f'{name} (%)', pad=0.01, shrink=0.8)

    ax.set_xticks(np.arange(len(hrus_c)))
    ax.set_yticks(np.arange(len(hrus_c)))
    ax.set_xticklabels([chr(ord('a') + i) for i in range(len(hrus_c))])
    ax.set_yticklabels([chr(ord('a') + i) for i in range(len(hrus_c))])

    for j in range(len(hrus_c)):
        ax.axhline(j - 0.5, color='k', lw=1.0)
        ax.axvline(j - 0.5, color='k', lw=1.0)

    ax.set_xlabel('')
    ax.set_ylabel('')

    ax.set_title(math, size=8)

    # ax.xaxis.set_ticks_position('top')
    # ax.xaxis.set_label_position('top')

    if i == 0:

        for i in range(len(subset)):
            extent = [i-0.5, i+0.5, i-0.5, i+0.5]
            bgr = mpl.patches.Rectangle(
                (i-0.5, i-0.5),  # bottom-left corner
                1,                    # width
                1,                    # height
                fill=False,
                edgecolor='k',
                linewidth=2,
                zorder=10
            )
            ax.add_patch(bgr)

        ax.annotate('Water %\nretained at \'a\'\n(learned retention $\mathbf{r}_t$)', color='k', xy=(0, 0), xytext=(-2, -2), size=8,
            horizontalalignment='center', arrowprops=dict(arrowstyle="-", color='tab:red',
            connectionstyle="angle3,angleA=-45,angleB=95"))

        ax.annotate(
            'Water %\ntransported from\n\'b\' to \'d\' (derived)', color='k', xy=(1, 3), xytext=(7, -2.5), size=8,
            horizontalalignment='center', arrowprops=dict(arrowstyle="-", color='tab:red',
            connectionstyle="angle3,angleA=110,angleB=-170"))



    else:

        ax.annotate(
            'Water % transported\nfrom \'b\' through \'d\' and\nfurther (derived)', color='k', xy=(1, 3), size=8,
            xytext=(-6, -3), horizontalalignment='center', arrowprops=dict(arrowstyle="-", color='tab:red',
            connectionstyle="angle3,angleA=-25,angleB=70"))

        ax.annotate(
            'Not connected:\n\'h\' and \'d\'', color='k', xy=(7, 3), xytext=(10, -1.2), size=8,
            horizontalalignment='center', arrowprops=dict(arrowstyle="-", color='tab:red',
            connectionstyle="angle3,angleA=-90,angleB=45"))

ax2.set_axis_off()

eq = '$\mathbf{q}_t = f_\mathrm{LSTM}(\mathbf{X}_{t-k:t}, \mathbf{Z})$\n'
eq += '$\mathbf{e}_t = f_\mathrm{evaoration}(\mathbf{T}_{\mathrm{air}, t}, \mathbf{w})$\n'
eq += '$\mathbf{S}\'_t=\max(\mathbf{S}_t+\mathbf{q}_t-\mathbf{e}_t, \mathbf{0})$\n'
eq += '$\mathbf{r}_t=\sigma(f_\mathrm{MLP}(\mathbf{Z}, \mathbf{S}_t))$\n'
eq += '$\mathbf{T}_t, \mathbf{D}_t=f_\mathrm{transport}(\mathbf{r}_t)$\n'
eq += '$\mathbf{S}_{t+1}=\mathbf{T}_t\, \mathbf{S}\'_t$\n'
eq += '$\mathbf{Q}_{t+1}=\mathbf{D}_t\, \mathbf{S}\'_t$'
ax0.text(1.1, 0.9, eq, transform=ax0.transAxes, fontsize=8, va='top', ha='left')

fig.savefig('./figures/concept.png', dpi=300, bbox_inches='tight')

In [ ]:
plt.rcParams["text.usetex"] = True
plt.rcParams["pgf.texsystem"] = "xelatex"

plt.rcParams["pgf.preamble"] = r"""
\usepackage{fontspec}
\setmonofont{Latin Modern Mono}  % this is the font used for \ttfamily
"""

plt.rcParams["font.family"] = "monospace"

In [ ]:
for i, (data, name, math, cmap) in enumerate(
        zip(
            [T, D], ['Transport', 'Discharge'], ['$\mathbf{T}_t$', '$\mathbf{D}_t$'], ['RdPu', 'GnBu']
        )
    ):
    
    fig, ax = plt.subplots(figsize=(3, 3), constrained_layout=True)

    nrows, ncols = data.shape
    extent = [-0.5, ncols - 0.5, -0.5, nrows - 0.5]

    axins = ax.inset_axes((0.83, 0.3, 0.02, 0.6))
    norm = mpl.colors.LogNorm(vmin=1e-2, vmax=1)

    im = ax.imshow(data, interpolation='none', norm=norm, cmap=cmap)

    mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cbar = plt.colorbar(
        mappable,
        cax=axins,
        orientation='vertical',   # or 'vertical'
        extend='min',
        label='Transport fraction (log scale)' if i == 0 else 'Discharge fraction (log scale)'
    )
    cbar_halo(cbar)

    ax.set_yticks(np.arange(0, len(hrus_c), 20))
    ax.set_xlabel('Drainage unit (DU)')
    ax.set_ylabel('Drainage unit (DU)')
    # ax.tick_params(axis='x', colors='w')
    # ax.set_yticks([])

    fig.savefig(f'./figures/concept_{name.lower()}.pdf', dpi=300, transparent=True)

In [ ]:
for i in range(2):

    fig, ax = plt.subplots(figsize=(3, 3), constrained_layout=True)

    axins = ax.inset_axes((0.8, 0.3, 0.02, 0.6))

    norm = mpl.colors.LogNorm(vmin=1e-2, vmax=1)
    cmap = 'RdPu'  if i == 0 else 'GnBu'
    var  = 'T_down' if i == 0 else 'D_down'
    data = ds_c[var].where(ds_c.T_down > 0).sel(k=slice(None, 110)).values
    im = ax.imshow(data, interpolation='none', norm=norm, cmap=cmap)

    mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cbar = plt.colorbar(
        mappable,
        cax=axins,
        orientation='vertical',   # or 'vertical'
        extend='min',
        label='Transport fraction (log scale)' if i == 0 else 'Discharge fraction (log scale)'
    )
    cbar_halo(cbar)

    ax.set_yticks(np.arange(0, len(hrus_c), 20))
    ax.set_xticks(np.arange(0, 101, 20))

    
    ax.set_title('')
    ax.set_ylabel('Drainage unit (DU)')
    ax.set_xlabel('Downstream-aligned DU ($k$)')

    name = 'transport_da' if i == 0 else 'discharge_da'
    fig.savefig(f'./figures/concept_{name.lower()}.pdf', dpi=300, transparent=True)

In [ ]:

vars = ['retention', 'height_mean', 'slope_mean', 'area', 'S_prime', 'S', 'Q', 'e', 'r']
cmaps = ['RdPu', 'Reds', 'Greens', 'Oranges', 'Blues', 'Blues', 'Blues', 'Blues', 'Blues']

for i, var in enumerate(vars):

    cmap = cmaps[i]
    fig, ax = plt.subplots(figsize=(3, 3), constrained_layout=True)
    
    mn, mx = hrus_c[var].quantile([0.0, 1.0]).values

    norm = mpl.colors.Normalize(vmin=mn, vmax=mx)
    hrus_c.plot(column=var, ax=ax, ec='k', lw=0.05, legend=False, cmap=cmap, norm=norm)
    ax.set_axis_off()

    fig.savefig(f'./figures/concept_{var.lower()}.pdf', dpi=300, transparent=True)

    fig, ax = plt.subplots(figsize=(0.5, 3), constrained_layout=True)
    vec = hrus_c[var].values.reshape(-1, 1)

    im = ax.imshow(vec, norm=norm, cmap=cmap)
    ax.set_xticks([])
    ax.set_yticks(np.arange(0, len(hrus_c), 20))
    ax.set_aspect(0.5)

    fig.savefig(f'./figures/concept_{var.lower()}_vec.pdf', dpi=300, transparent=True)


## XVAL sets

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 5), subplot_kw={'projection': lv95}, gridspec_kw={'wspace': 0.02})

for i, (ax, title, xval_dir) in enumerate(zip(axes, ['Random split: RS', 'Catchment split (easy): CS-E', 'Catchment split (hard): CS-H'], [
        '/net/argon/landclim/kraftb/machflow_routing/logs/routed/random/xval',
        '/net/argon/landclim/kraftb/machflow_routing/logs/routed/easy/xval',
        '/net/argon/landclim/kraftb/machflow_routing/logs/routed/hard/xval'
    ])):

    # if i == 1:
    #     break

    color_mapping = {
        'None': '#FFFFFF',
        'Training': '#FFC107',
        'Validation': '#1E88E5',
        'Test': '#D81B60'
    }

    poly = hrus.copy()

    paths = glob(os.path.join(xval_dir, 'fold_*/preds.zarr'))

    rivers_subset.plot(
        ec='k', fc='none', linewidth=(np.log(rivers_subset.CUM_LEN + 1) / 10 - 0.5),
        alpha=1.0,
        zorder=2,
        ax=ax)

    for path in paths:
        ds = xr.open_zarr(path)
        poly['set'] = ''
        poly.loc[ds.hru, 'set'] = [list(color_mapping.values())[v] for v in ds.subset.values]

        if i == 0:
            color = 'none'
        else:
            color = poly['set']
        poly.plot(ec='k', lw=0.12, legend=False, color=color, alpha=0.8, zorder=1, ax=ax)
        for _, s in enumerate([1, 2, 3]):
            hru_subset = ds.hru.where((ds.subset.load() == s) & ds.has_obs.load(), drop=True).values
            poly.loc[hru_subset].centroid.plot(
                ec='k',
                fc=list(color_mapping.values())[s],
                lw=0.9,
                markersize=10,
                alpha=0.8,
                zorder=3,
                ax=ax)

        despine(ax=ax, which='all', remove_ticks=True)

        ax.set_title(title)

        if i == 1:
            # Create legend patches
            legend_patches = [mpl.patches.Patch(ec='k', fc=color, label=category, lw=0.15) for category, color in color_mapping.items()]

            # Add the legend to the plot
            legend = ax.legend(handles=legend_patches, title='Data split', frameon=False, loc='upper center',
                               bbox_to_anchor=(0.5, 0), fontsize=8, title_fontsize=8, ncol=4)

fig.savefig('./figures/cv_sets.png', dpi=300, bbox_inches='tight')
